# Exercice Pinecone : Reranking et Recherche Sémantique Médicale

Ce notebook est conçu pour vous guider à travers l'utilisation de Pinecone pour le reranking (réordonnancement) de documents et la mise en place d'un index serverless pour des notes médicales.

## Partie 1 : Chargement des Documents et Modèle de Reranking

### 1. Installation des bibliothèques Pinecone

In [ ]:
# Installation des packages nécessaires pour interagir avec l'API Pinecone
!pip install -U pinecone==6.0.1 pinecone-notebooks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.4/421.4 kB 17.2 MB/s eta 0:00:00


### 2. Authentification avec Pinecone

Nous utilisons le helper `pinecone-notebooks` pour une connexion sécurisée sans exposer la clé API en clair.

In [ ]:
import os
# Vérifie si la clé est déjà dans l'environnement, sinon lance l'outil d'authentification
if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

### 3. Instanciation du client Pinecone

In [ ]:
from pinecone import Pinecone
import os

# Tentative de récupération de la clé API
api_key = os.environ.get("PINECONE_API_KEY")

if not api_key:
    print("Erreur : La clé API Pinecone n'est pas définie. Veuillez exécuter la cellule d'authentification ci-dessus.")
else:
    # Initialisation du client
    pc = Pinecone(api_key=api_key)
    print("Client Pinecone initialisé avec succès.")

Client Pinecone initialisé avec succès.


### 4. Définition de la requête et des documents

Nous allons tester la capacité du modèle à distinguer "Apple" (l'entreprise) du fruit.

In [ ]:
query = "Tell me about Apple's products"

documents = [
    "The Granny Smith is a popular green apple variety with a tart flavor.", # Fruit
    "The iPhone 15 Pro features a titanium design and the A17 Pro chip.", # Entreprise
    "Apples are high in fiber and vitamin C, making them a healthy snack.", # Fruit
    "The MacBook Air with M3 chip offers exceptional performance and battery life.", # Entreprise
    "Apple's latest Vision Pro headset introduces spatial computing to the market." # Entreprise
]

### 5. Appel au Reranker

Le reranking permet d'affiner les résultats en utilisant un modèle plus puissant sur un sous-ensemble de documents.

In [ ]:
from pinecone import RerankModel

# Utilisation du modèle bge-reranker pour réordonner les documents
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3
)
print("Reranking réussi.")

Reranking réussi.


### 6. Inspection des résultats

In [ ]:
def show_reranked_results(query, matches):
    print(f"Requête : {query}\n")
    for i, m in enumerate(matches):
        # Accès correct à l'attribut .score et au texte du document
        print(f"Rang {i+1} (Score: {m.score:.4f}) : {m.document['text']}")

# L'attribut contenant les résultats de l'inférence est .data
show_reranked_results(query, reranked.data)

Requête : Tell me about Apple's products

Rang 1 (Score: 0.0538) : Apple's latest Vision Pro headset introduces spatial computing to the market.
Rang 2 (Score: 0.0486) : Apples are high in fiber and vitamin C, making them a healthy snack.
Rang 3 (Score: 0.0245) : The MacBook Air with M3 chip offers exceptional performance and battery life.


## Partie 2 : Configuration d'un Index Serverless pour Notes Médicales

In [ ]:
# Installation des bibliothèques de manipulation de données et de Deep Learning
!pip install pandas torch transformers

In [ ]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Configuration de l'environnement Pinecone
cloud = os.getenv('PINECONE_CLOUD', 'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')

# Spécification serverless
spec = ServerlessSpec(cloud=cloud, region=region)

# Nom de l'index
index_name = 'medical-notes-index'

### Création de l'index

In [ ]:
# Nettoyage : suppression de l'index s'il existe déjà
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# Création de l'index avec une dimension de 384 (pour le modèle all-MiniLM-L6-v2)
pc.create_index(
    name=index_name,
    dimension=384,
    metric='cosine',
    spec=spec
)
print(f"Index '{index_name}' créé.")

Index 'medical-notes-index' créé.


## Partie 3 : Chargement des Données

Nous téléchargeons un fichier JSONL contenant des notes médicales pré-traitées.

In [ ]:
import requests
import tempfile
import os
import pandas as pd

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # Correction de l'URL vers la version correcte du dépôt Pinecone
    url = "https://raw.githubusercontent.com/pinecone-io/examples/master/docs/data/sample_notes_data.jsonl"

    print(f"Téléchargement depuis : {url}")
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    # Lecture du fichier JSONL
    df = pd.read_json(file_path, orient='records', lines=True)

print("Données chargées avec succès.")
print("Dimensions :", df.shape)
display(df.head())

Téléchargement depuis : https://raw.githubusercontent.com/pinecone-io/examples/master/docs/data/sample_notes_data.jsonl
Données chargées avec succès.
Dimensions : (100, 3)


,id,values,metadata
0,P011,"[-0.2027486265, 0.2769146562, -0.1509393603, 0...","{'advice': 'rest, hydrate', 'symptoms': 'heada..."
1,P001,"[0.1842793673, 0.4459365904, -0.0770567134, 0....","{'tests': 'EKG, stress test', 'symptoms': 'che..."
2,P002,"[-0.2040648609, -0.1739618927, -0.2897160649, ...","{'HbA1c': '7.2', 'condition': 'diabetes', 'med..."
3,P003,"[0.1889383644, 0.2924542725, -0.2335938066, -0...","{'symptoms': 'cough, wheezing', 'diagnosis': '..."
4,P004,"[-0.12171068040000001, 0.1674752235, -0.231888...","{'referral': 'dermatology', 'condition': 'susp..."


## Partie 4 : Upsert (Insertion) des Données

In [ ]:
# Connexion à l'index spécifique
index = pc.Index(name=index_name)

# Insertion des données directement depuis le DataFrame
index.upsert_from_dataframe(df)
print("Upsert terminé.")

sending upsert requests:   0%|          | 0/100 [00:00<?, ?it/s]

Upsert terminé.


In [ ]:
import time

def is_fresh(index_obj):
    stats = index_obj.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Nombre de vecteurs : {vector_count}")
    return vector_count > 0

# On s'assure que l'objet index est bien initialisé avant la boucle
index = pc.Index(name=index_name)

# Attente de la disponibilité des données indexées
while not is_fresh(index):
    time.sleep(5)

print("Index prêt !")
display(index.describe_index_stats())

Nombre de vecteurs : 100
Index prêt !


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 100}},
 'total_vector_count': 100,
 'vector_type': 'dense'}

## Partie 5 : Recherche Sémantique

### 1. Fonction d'embedding

In [ ]:
def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')

    with torch.no_grad():
        model_output = model(**encoded_input)
        # On moyenne sur la dimension 1 (sequence length) pour obtenir un vecteur unique
        embedding = model_output.last_hidden_state[0].mean(dim=0)
    return embedding

### 2. Exécution d'une requête

In [ ]:
question = "patient with severe chest pain and shortness of breath"
query_vector = get_embedding(question).tolist()

# Recherche des 5 résultats les plus proches
results = index.query(vector=[query_vector], top_k=5, include_metadata=True)

# Tri par score décroissant
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)
print("Recherche sémantique terminée.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Recherche sémantique terminée.


## Partie 6 : Affichage et Reranking Clinique

In [ ]:
def show_results(question, matches):
    print(f"Question: '{question}'")
    print('\nRésultats initiaux (Top similarity) :')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f'   Score: {match["score"]:.4f}')
        print(f'   Métadonnées: {match["metadata"]}')
        print('')

# Utilisation des résultats triés obtenus à l'étape précédente
show_results(question, sorted_matches)

Question: 'patient with severe chest pain and shortness of breath'

Résultats initiaux (Top similarity) :
   1. ID: P001
   Score: 0.6338
   Métadonnées: {'symptoms': 'chest pain', 'tests': 'EKG, stress test'}

   2. ID: P016
   Score: 0.4918
   Métadonnées: {'condition': 'heart murmur', 'referral': 'cardiology'}

   3. ID: P003
   Score: 0.4615
   Métadonnées: {'diagnosis': 'bronchitis', 'symptoms': 'cough, wheezing', 'treatment': 'antibiotics'}

   4. ID: P063
   Score: 0.4240
   Métadonnées: {'diagnosis': 'pneumonia', 'symptoms': 'cough, fever', 'treatment': 'antibiotics'}

   5. ID: P0100
   Score: 0.4054
   Métadonnées: {'advice': 'over-the-counter pain relief, stretching', 'symptoms': 'muscle pain'}



### Préparation pour le Reranking

In [ ]:
# On s'assure que 'results' provient bien de l'exécution de la recherche sémantique
# Transformation des résultats pour le module Inference de Pinecone
transformed_documents = [
    {
        'id': match['id'],
        # On combine les métadonnées en une chaîne de caractères pour le reranker
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]
print(f"{len(transformed_documents)} documents préparés pour le reranking.")

5 documents préparés pour le reranking.


### Exécution du Reranking final

In [ ]:
refined_query = "emergency cardiology assessment for chest pain"

# Exécution du reranking sur les documents transformés
# Utilise transformed_documents défini à l'étape précédente
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True
)
print("Reranking médical terminé avec succès.")

Reranking médical terminé avec succès.


In [ ]:
def show_reranked_results_final(question, matches):
    print(f"Requête affinée: '{question}'")
    print('\nRésultats finaux après Reranking :')
    for i, match in enumerate(matches):
        # Utilisation des attributs de l'objet Match (document.id, score, document.reranking_field)
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f'   Score de Reranking: {match.score:.4f}')
        print(f'   Contenu traité: {match.document.reranking_field}')
        print('')

# Affichage des données du résultat de reranking
show_reranked_results_final(refined_query, reranked_results.data)

Requête affinée: 'emergency cardiology assessment for chest pain'

Résultats finaux après Reranking :
   1. ID: P001
   Score de Reranking: 0.1984
   Contenu traité: symptoms: chest pain; tests: EKG, stress test

   2. ID: P016
   Score de Reranking: 0.0046
   Contenu traité: condition: heart murmur; referral: cardiology

   3. ID: P0100
   Score de Reranking: 0.0012
   Contenu traité: advice: over-the-counter pain relief, stretching; symptoms: muscle pain



### Nettoyage

Il est important de supprimer l'index pour éviter des coûts inutiles.

In [ ]:
# Suppression de l'index
pc.delete_index(name=index_name)
print("Index supprimé.")

Index supprimé.
